In [2]:
import requests, re, json
from bs4 import BeautifulSoup

def findTotalRecords(soup):
    # Find the <script> tag containing the totalRecords
    script_tag = soup.find('script', string=re.compile('totalRecords'))

    total_records = 0
    if script_tag:
        match = re.search(r'totalRecords\s*:\s*(\d+)', script_tag.string)
        if match: total_records = int(match.group(1))
    
    return total_records

def fetch(url):

    session = requests.Session() # use same session to allow page switching

    response = session.get(url)
    content = response.content
    
    total_records = findTotalRecords(BeautifulSoup(content, "lxml"))
    if total_records == 300: print("NOTE: MAX RECORD COUNT HIT WITH URL", url)

    # 0-50 records means no additional pages, 51-100 means 1 additional page, etc.
    additional_pages = max(0, total_records // 50 - (not (total_records % 50))) # note 0 gives -1
    pageUrl = "https://more.app.vanderbilt.edu/more/SearchClassesExecute!switchPage.action?pageNum="

    for i in range(additional_pages):
        add_response = session.get(pageUrl + str(i + 2))
        content += add_response.content
    
    soup = BeautifulSoup(content, "lxml")
    return soup

In [3]:
def scrape_course_listings(soup):
    # find all <td> with "classNumber_" in id
    listing_elements = soup.find_all('td', id=lambda x: x and x.startswith("classNumber_"))

    new_data = []
    for listing in listing_elements:
        onclick_text = listing.get('onclick', '')
        
        class_number, term_code = None, None
        if "classNumber" in onclick_text and "termCode" in onclick_text:
            class_number = onclick_text.split("classNumber : '")[1].split("'")[0]
            term_code = onclick_text.split("termCode : '")[1].split("'")[0]
        
        if class_number and term_code:
            new_data.append({'classNumber': class_number, 'termCode': term_code})
    
    return new_data

def save_course_listings(new_data):

    # load existing data
    try:
        with open('course_listings.json', 'r') as file:
            existing_data = json.load(file)
    except FileNotFoundError:
        existing_data = []

    # duplicate detection
    existing_set = set((entry["classNumber"], entry["termCode"]) for entry in existing_data)

    # add unique entries
    for entry in new_data:
        pair = (entry["classNumber"], entry["termCode"])
        if pair not in existing_set:
            existing_data.append(entry)
            existing_set.add(pair)

    with open('course_listings.json', 'w') as file:
        json.dump(existing_data, file, indent=4)

In [4]:
# scraping course listings
from tqdm import tqdm

base_url = "https://more.app.vanderbilt.edu/more/SearchClassesExecute!search.action?keywords="

edges = [100, 110, 385, 799, 850, 899] # keywords that have over 300 entries (gets truncated)
# skip *999 series since it's mostly phd dissertation research (7999, 8999, 9999 have 300+ each)
for i in tqdm(range(999), desc=f"Processing keywords", unit="keyword"):

    if i in edges:
        addons = [str(j) + f"{i:03d}" for j in range(10)]
    else:
        addons = [f"{i:03d}"]

    for addon in addons:
        url = base_url + addon

        response = requests.get(url)
        content = response.content

        total_records = findTotalRecords(BeautifulSoup(content, "lxml"))

        try:
            soup = fetch(url)
            new_data = scrape_course_listings(soup)
            save_course_listings(new_data)
        except:
            print(f"error with keyword '{addon}'")

Processing keywords': 100%|██████████| 999/999 [26:28<00:00,  1.59s/keyword]  


In [77]:
# Load JSON file
with open('classNumber_termCode_data.json', 'r') as file:
    data = json.load(file)

print(data)
# Access data by index
print(data[0]['classNumber'], data[0]['termCode'])

[{'classNumber': '2253', 'termCode': '1040'}, {'classNumber': '2846', 'termCode': '1040'}, {'classNumber': '1023', 'termCode': '1040'}, {'classNumber': '2523', 'termCode': '1040'}, {'classNumber': '9648', 'termCode': '1040'}, {'classNumber': '1563', 'termCode': '1040'}, {'classNumber': '10012', 'termCode': '1040'}, {'classNumber': '1558', 'termCode': '1040'}, {'classNumber': '1559', 'termCode': '1040'}, {'classNumber': '1560', 'termCode': '1040'}, {'classNumber': '10839', 'termCode': '1040'}, {'classNumber': '2333', 'termCode': '1040'}, {'classNumber': '2334', 'termCode': '1040'}, {'classNumber': '4739', 'termCode': '1040'}, {'classNumber': '1117', 'termCode': '1040'}, {'classNumber': '3169', 'termCode': '1040'}, {'classNumber': '1116', 'termCode': '1040'}, {'classNumber': '10052', 'termCode': '1040'}, {'classNumber': '10054', 'termCode': '1040'}, {'classNumber': '10055', 'termCode': '1040'}, {'classNumber': '4355', 'termCode': '1040'}, {'classNumber': '4438', 'termCode': '1040'}, {'cl

In [6]:
def build_class_data(class_number, course_dept, course_code, class_section, course_title,
                     school, career, class_type, credit_hours, grading_basis, consent, term_year,
                     term_season, session, dates, requirements, description, notes, capacity, enrolled,
                     wl_capacity, wl_occupied, status, attributes, meetings, instructors):
    current_data = {
        "class_number": class_number,
        "course_dept": course_dept,
        "course_code": course_code,
        "class_section": class_section,
        "course_title": course_title,
        "school": school,
        "career": career,
        "class_type": class_type,
        "credit_hours": credit_hours,
        "grading_basis": grading_basis,
        "consent": consent,
        "term_year": term_year,
        "term_season": term_season,
        "session": session,
        "dates": dates,
        "requirements": requirements,
        "description": description,
        "notes": notes,
        "availability": {
            "capacity": capacity,
            "enrolled": enrolled,
            "wl_capacity": wl_capacity,
            "wl_occupied": wl_occupied,
            "status": status,
        },
        "attributes": attributes,
        "meeting": [build_meeting_data(meeting) for meeting in meetings],  # Using the helper for meetings
        "instructors": instructors,
    }
    return current_data

def build_meeting_data(meeting):
    return {
        "meeting_days": meeting.get("meeting_days", None),
        "meeting_time": meeting.get("meeting_time", None),
        "meeting_dates": meeting.get("meeting_dates", None),
    }

In [7]:
def extract_class_details(soup):
    header = soup.find("h1").text.strip()
    parts = header.split(":")
    dept_code, section_and_title = parts[0].split("-"), parts[1].strip()

    course_dept, course_code, class_section = dept_code[0], dept_code[1], dept_code[2]
    course_title = section_and_title.split(" ", 1)[1]
    return course_dept, course_code, class_section, course_title

def extract_other_details(soup):
    details_table = soup.find("table", class_="nameValueTable")
    left_table, right_table = details_table.find_all("table")
    
    # Left-side details
    school = left_table.find("td", text="School:").find_next_sibling("td").text.strip()
    career = left_table.find("td", text="Career:").find_next_sibling("td").text.strip()
    class_type = left_table.find("td", text="Component:").find_next_sibling("td").text.strip()
    credit_hours = left_table.find("td", text="Hours:").find_next_sibling("td").text.strip()
    grading_basis = left_table.find("td", text="Grading Basis:").find_next_sibling("td").text.strip()
    consent = left_table.find("td", text="Consent:").find_next_sibling("td").text.strip()

    # Right-side details
    term = right_table.find("td", text="Term:").find_next_sibling("td").text.strip()
    term_season, term_year = term.split(" ")
    session = right_table.find("td", text="Session:").find_next_sibling("td").text.strip()
    dates = right_table.find("td", text="Session Dates:").find_next_sibling("td").text.strip()
    requirements = right_table.find("td", text="Requirement(s):").find_next_sibling("td").text.strip()

    return school, career, class_type, credit_hours, grading_basis, consent, term_year, term_season, session, dates, requirements

def extract_description(soup):
    desc_div = soup.find("div", class_="detailPanel")
    description = desc_div.text.strip()
    return description

def extract_availability(soup):
    availability_table = soup.find("table", class_="availabilityNameValueTable")
    capacity = availability_table.find("td", text="Class Capacity:").find_next_sibling("td").text.strip()
    enrolled = availability_table.find("td", text="Total Enrolled:").find_next_sibling("td").text.strip()
    wl_capacity = availability_table.find("td", text="Wait List Capacity:").find_next_sibling("td").text.strip()
    wl_occupied = availability_table.find("td", text="Total on Wait List:").find_next_sibling("td").text.strip()
    status = soup.find("div", class_="availabiltyIndicator").find("span", class_="open").text.strip()

    return {
        "capacity": capacity,
        "enrolled": enrolled,
        "wl_capacity": wl_capacity,
        "wl_occupied": wl_occupied,
        "status": status,
    }

def extract_attributes(soup):
    attributes_div = soup.find("div", class_="detailPanel", text="Attributes")
    attributes = [item.text.strip() for item in attributes_div.find_all("div", class_="listItem")]
    return attributes

def extract_meetings_and_instructors(soup):
    meeting_table = soup.find("div", class_="detailPanel").find("table", class_="meetingPatternTable")
    meetings = []
    
    for row in meeting_table.find_all("tr")[1:]:
        columns = row.find_all("td")
        meeting_days = columns[0].text.strip()
        meeting_time = columns[1].text.strip()
        meeting_dates = columns[3].text.strip()
        instructors = [inst.text.strip() for inst in columns[4].find_all("div")]
        meetings.append({
            "meeting_days": meeting_days,
            "meeting_time": meeting_time,
            "meeting_dates": meeting_dates,
            "instructors": instructors,
        })
    return meetings

In [8]:
def scrape_data(soup):

    # Extract details
    class_number = soup.find("div", class_="classNumber").text.split(":")[1].strip()
    course_dept, course_code, class_section, course_title = extract_class_details(soup)
    school, career, class_type, credit_hours, grading_basis, consent, term_year, term_season, session, dates, requirements = extract_other_details(soup)
    description = extract_description(soup)
    availability = extract_availability(soup)
    attributes = extract_attributes(soup)
    meetings = extract_meetings_and_instructors(soup)

    # Build data dictionary
    current_data = {
        "class_number": class_number,
        "course_dept": course_dept,
        "course_code": course_code,
        "class_section": class_section,
        "course_title": course_title,
        "school": school,
        "career": career,
        "class_type": class_type,
        "credit_hours": credit_hours,
        "grading_basis": grading_basis,
        "consent": consent,
        "term_year": term_year,
        "term_season": term_season,
        "session": session,
        "dates": dates,
        "requirements": requirements,
        "description": description,
        "notes": None,  # Not in provided HTML
        "availability": availability,
        "attributes": attributes,
        "meeting": meetings,
    }

    return current_data

In [9]:
url = "https://more.app.vanderbilt.edu/more/GetClassSectionDetail.action?classNumber=4755&termCode=1040"
soup = fetch(url)

data = scrape_data(soup)
print(data)

/var/folders/dc/hffdq1tn1477q1q4sytpyg6h0000gn/T/ipykernel_65796/548204570.py:15: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  school = left_table.find("td", text="School:").find_next_sibling("td").text.strip()
/var/folders/dc/hffdq1tn1477q1q4sytpyg6h0000gn/T/ipykernel_65796/548204570.py:16: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  career = left_table.find("td", text="Career:").find_next_sibling("td").text.strip()
/var/folders/dc/hffdq1tn1477q1q4sytpyg6h0000gn/T/ipykernel_65796/548204570.py:17: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  class_type = left_table.find("td", text="Component:").find_next_sibling("td").text.strip()
/var/folders/dc/hffdq1tn1477q1q4sytpyg6h0000gn/T/ipykernel_65796/548204570.py:18: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  credi

AttributeError: 'NoneType' object has no attribute 'find_all'

In [20]:
# parsing data (based off code block above)

import json
from datetime import datetime

data = []

# course codes (ex. CS 1101)
course_code_elements = soup.find_all('span', class_='classAbbreviation')

course_code_elements = course_code_elements[18]
for course_code in course_code_elements:

    # ex: Computer Science
    subject_name = course_code.find_previous('div', class_='subjectHeader')
    # name, number = course_code.get_text(strip=True).rstrip(':').split(' ')
    # number = int(number) # edge cases: 1600L, 2500D

    # ex: Programming and Problem Solving
    course_name = course_code.find_next('span', class_='classDescription')

    class_section = course_name.find_next('td', class_='classSection')

    credit_hours = class_section.find_next('td', class_='classHours')

    class_type = credit_hours.find_next('td', class_='classType')

    availability = class_type.find_next('a')
    if "availableStatus" in availability.get('class'):
        status = 0 # note that we NEVER get here if waitlist is nonempty
    elif "waitlistStatus" in availability.get('class'):
        status = 1
    elif "closedStatus" in availability.get('class'):
        status = 2
    else:
        # availability.get() returns a list
        raise ValueError(f"Class availability status unaccounted for: {availability.get('class')}")
    
    occupied, capacity = map(int, availability.get_text(strip=True).split('/'))

    # can be TBA
    meeting_days = availability.find_next('td', class_='classMeetingDays')

    # TODO: edge case: TBA, multiple listings
    meeting_times = meeting_days.find_next('td', class_='classMeetingTimes')

    # # Find all time slots (09:00a - 12:00p) and corresponding dates (02/02/2025 - 02/02/2025)
    # times = meeting_times.find_all(string=lambda text: text and 'a - p' in text)
    # dates = meeting_times.find_all('div', class_='randomDates')

    # # Extract the time and date pairs
    # time_date_pairs = []
    # for time, date in zip(times, dates):
    #     time_date_pairs.append((time.strip(), date.get_text().strip()))

    # # Print the pairs
    # for pair in time_date_pairs:
    #     print(pair)
    print(meeting_times)
    """
    # convert meeting time to 24-hr numerical values
    meeting_times_str = meeting_times.get_text(strip=True).split('\n')[0]
    start_time_str, end_time_str = meeting_times_str.strip().upper().split(' - ')
    start_time = datetime.strptime(start_time_str + "M", '%I:%M%p').strftime('%H:%M')
    end_time = datetime.strptime(end_time_str + "M", '%I:%M%p').strftime('%H:%M')
    
    date_range = meeting_times.find('div', class_='randomDates')
    
    # NOTE: this is usually dynamically loaded in so will likely be None
    location = meeting_times.find_next('td', class_='classBuilding')
    """
    location = meeting_days.find_next('td', class_='classBuilding')

    instructors = location.find_next('td', class_='classInstructor')

    # checking for note
    class_action_dummy = instructors.find_next('td', class_='classActions')
    next_sibling = class_action_dummy.find_next('tr')
    note = None
    if next_sibling and next_sibling.get('id', '').endswith('_notes'):
        note = next_sibling.find('td', colspan='9').get_text().strip()


    current_data = {
        "subject_name" : subject_name.find('h4').get_text(strip=True),
        # "course_code_name" : name,
        # "course_code_number" : number,
        "course_code" : course_code.get_text(strip=True).rstrip(":"),
        "course_name" : course_name.get_text(strip=True),
        "class_section" : class_section.get_text(strip=True),
        "credit_hours" : float(re.sub(r'[^\d.]+', '', credit_hours.get_text(strip=True))),
        "class_type" : class_type.get_text(strip=True),
        "availability": {
            "status": status,
            "occupied": occupied,
            "capacity": capacity
        },
        "meeting_days" : meeting_days.get_text(strip=True).replace('<br/>', ''),
        "meeting_start" : start_time,
        "meeting_end" : end_time,
        "date_range" : date_range.get_text(strip=True) if date_range else None,
        "location" : location.get_text(strip=True) if location.get_text(strip=True) else None,
        "instructors" : [instructor.strip() for instructor in instructors.get_text(strip=True).split('|')],
        "note" : note
    }

    data.append(current_data)

filename = "data.json"
with open(filename, "w") as file:
    json.dump(data, file, indent=4)

<td class="classMeetingTimes" onclick="YAHOO.mis.student.Topics.showClassDetailPanel.fire({classNumber : '10582', termCode : '1040', hideAddToCartButton : 'false'})">
         
            
               07:15p - 09:15p<br/>
<div class="randomDates">01/30/2025 - 01/30/2025</div>
               
            
               04:30p - 07:30p<br/>
<div class="randomDates">01/31/2025 - 01/31/2025</div>
               
            
               09:00a - 12:00p<br/>
<div class="randomDates">02/01/2025 - 02/01/2025</div>
               
            
               01:00p - 04:00p<br/>
<div class="randomDates">02/01/2025 - 02/01/2025</div>
               
            
               09:00a - 12:00p<br/>
<div class="randomDates">02/02/2025 - 02/02/2025</div>
</td>
